In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 

In [ ]:
%pwd

In [ ]:
STATISTICAL_ANALYSIS_PATH = "/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/clinical_data_analysis/statistical_analysis/all_high_vs_low_stats_all_biomarkers_0444.csv"
DATA_PATH = "/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/clinical_data_analysis/statistical_analysis/Tau/217/high_vs_low_data.csv"

In [ ]:
# find the ID of the patient with the highest trail b score
df_high = pd.read_csv(DATA_PATH)
df_high = df_high[~(df_high['Trail_B_(second)'] == df_high['Trail_B_(second)'].max())]
df_high.head(1)

In [ ]:
df = pd.read_csv(STATISTICAL_ANALYSIS_PATH)
print(df.shape)
df.head()

In [ ]:
df_tau = df[df['Biomarker_Type'] == 'tau_217']
df_tau_cog = df_tau[df_tau['Source'] == 'cognitive']
df_tau_cog_all = df_tau_cog[df_tau_cog['MCI_Status'] == 'All']
df_tau_cog_all.head(1)

# Cognitive tests visualization

In [ ]:
df_tau = df[df['Biomarker_Type'] == 'tau_217']
df_tau_cog = df_tau[df_tau['Source'] == 'cognitive']
df_tau_cog_all = df_tau_cog[df_tau_cog['MCI_Status'] == 'All']
df_tau_cog_all.head(1)

In [ ]:
# Map feature display names to column names in df_high
feature_col_map = {
    "MOCA score": "MOCA_score",
    "TMT (Trail B)": "Trail_B_(second)",
    "IST": "IST"
}

# Get adjusted p-values from df_tau_cog_all
p_values_dict = {}
for feature_name in feature_col_map.keys():
    feature_row = df_tau_cog_all[df_tau_cog_all['Feature'] == feature_name]
    if not feature_row.empty:
        p_values_dict[feature_name] = feature_row['adjusted p-value'].values[0]
    else:
        p_values_dict[feature_name] = None

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot violin plots for each feature
for idx, (feature_name, col_name) in enumerate(feature_col_map.items()):
    ax = axes[idx]
    
    # Filter data to remove NaN values for this feature
    plot_data = df_high[df_high[col_name].notna()].copy()
    
    # Create violin plot
    sns.violinplot(
        data=plot_data,
        x='tau_217_category',
        y=col_name,
        ax=ax,
        palette=['#b4cde3', '#fbb4ae']  # blue for Low, red for High
    )
    
    # Add significance marker with adjusted p-value
    p_val = p_values_dict[feature_name]
    if p_val is not None:
        # Determine significance level
        if p_val < 0.001:
            sig_marker = '***'
        elif p_val < 0.01:
            sig_marker = '**'
        elif p_val < 0.05:
            sig_marker = '*'
        else:
            sig_marker = 'ns'
        
        # Add text above the violins (using data coordinates)
        y_max = plot_data[col_name].max()
        y_min = plot_data[col_name].min()
        y_range = y_max - y_min
        y_position = y_max + (y_range * 0.05)  # Position 5% above max value
        
        # Position x at 0.5 (between Low=0 and High=1 in categorical space)
        ax.text(0.5, y_position, f'{sig_marker}\np = {p_val:.4f}', 
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    
    ax.set_title(feature_name, fontsize=14, fontweight='bold')
    ax.set_xlabel('p-tau217 Category', fontsize=12)
    ax.set_ylabel(feature_name, fontsize=12)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Extract adjusted p-values for the three cognitive features
features = ["MOCA score", "TMT (Trail B)", "IST"]
p_values = []
feature_labels = []

for feature_name in features:
    feature_row = df_tau_cog_all[df_tau_cog_all['Feature'] == feature_name]
    if not feature_row.empty:
        p_val = feature_row['adjusted p-value'].values[0]
        p_values.append(p_val)
        feature_labels.append(feature_name)

# Compute log(1 + adjusted p-values) for plotting
log_p_values = [-1 * np.log(p) for p in p_values]

# Calculate log(1 + threshold) values for lines
thresholds = [0.05, 0.01, 0.001]
log_thresholds = [-1 * np.log(thr) for thr in thresholds]

# Create bar plot
fig, ax = plt.subplots(figsize=(10, 6))

# Create bars with color coding based on significance (same coloring as before)
colors = []
for p_val in p_values:
    if p_val < 0.001:
        colors.append('#d62728')  # red for highly significant
    elif p_val < 0.01:
        colors.append('#ff7f0e')  # orange for very significant
    elif p_val < 0.05:
        colors.append('#2ca02c')  # green for significant
    else:
        colors.append('#7f7f7f')  # gray for not significant

bars = ax.bar(feature_labels, log_p_values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

# Add ORIGINAL p-value text on the bars for interpretability (not the log value)
for bar, p_val in zip(bars, p_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height/2,
            f'p = {p_val:.4f}', ha='center', va='center', 
            fontsize=11, fontweight='bold', color='white')

# Add log-threshold lines (with original threshold in the label)
ax.axhline(y=log_thresholds[0], color='red', linestyle='--', linewidth=2, alpha=0.5, label='p = 0.05 threshold')
ax.axhline(y=log_thresholds[1], color='orange', linestyle='--', linewidth=2, alpha=0.5, label='p = 0.01 threshold')
ax.axhline(y=log_thresholds[2], color='darkred', linestyle='--', linewidth=2, alpha=0.5, label='p = 0.001 threshold')

ax.set_ylabel('Negative log(Adjusted p-value)', fontsize=12, fontweight='bold')
ax.set_xlabel('Cognitive Feature', fontsize=12, fontweight='bold')
ax.set_title('Negative log(Adjusted p-values) for Cognitive Features\n(High vs Low p-tau217)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(0, max(log_p_values + log_thresholds) * 1.2)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.legend(loc='upper right', fontsize=10)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Map feature display names to column names in df_high
feature_col_map = {
    "MOCA score": "MOCA_score",
    "TMT (Trail B)": "Trail_B_(second)",
    "IST": "IST"
}

# Get adjusted p-values from df_tau_cog_all
p_values_dict = {}
for feature_name in feature_col_map.keys():
    feature_row = df_tau_cog_all[df_tau_cog_all['Feature'] == feature_name]
    if not feature_row.empty:
        p_values_dict[feature_name] = feature_row['adjusted p-value'].values[0]
    else:
        p_values_dict[feature_name] = None

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot violin plots for each feature
for idx, (feature_name, col_name) in enumerate(feature_col_map.items()):
    ax = axes[idx]
    
    # Filter data to remove NaN values for this feature
    plot_data = df_high[df_high[col_name].notna()].copy()
    
    # Create violin plot
    sns.boxplot(
        data=plot_data,
        x='tau_217_category',
        y=col_name,
        ax=ax,
        palette=['#b4cde3', '#fbb4ae']  # blue for Low, red for High
    )
    
    # Add significance marker with adjusted p-value
    p_val = p_values_dict[feature_name]
    if p_val is not None:
        # Determine significance level
        if p_val < 0.001:
            sig_marker = '***'
        elif p_val < 0.01:
            sig_marker = '**'
        elif p_val < 0.05:
            sig_marker = '*'
        else:
            sig_marker = 'ns'
        
        # Add text above the violins (using data coordinates)
        y_max = plot_data[col_name].max()
        y_min = plot_data[col_name].min()
        y_range = y_max - y_min
        y_position = y_max - (y_range * 0.05)  # Position 5% above max value
        
        # Position x at 0.5 (between Low=0 and High=1 in categorical space)
        ax.text(0.5, y_position, f'{sig_marker}\np = {p_val:.4f}', 
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    
    ax.set_title(feature_name, fontsize=14, fontweight='bold')
    ax.set_xlabel('p-tau217 Category', fontsize=12)
    ax.set_ylabel(feature_name, fontsize=12)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Map feature display names to column names in df_high
feature_col_map = {
    "MOCA score": "MOCA_score",
    "TMT (Trail B)": "Trail_B_(second)",
    "IST": "IST"
}

# Get adjusted p-values from df_tau_cog_all
p_values_dict = {}
for feature_name in feature_col_map.keys():
    feature_row = df_tau_cog_all[df_tau_cog_all['Feature'] == feature_name]
    if not feature_row.empty:
        p_values_dict[feature_name] = feature_row['adjusted p-value'].values[0]
    else:
        p_values_dict[feature_name] = None

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot violin plots for each feature
for idx, (feature_name, col_name) in enumerate(feature_col_map.items()):
    ax = axes[idx]
    
    # Filter data to remove NaN values for this feature
    plot_data = df_high[df_high[col_name].notna()].copy()
    
    # Density histograms: each group (Low/High) is normalized to area=1,
    # so peak height depends on spread—narrower distribution → higher peak, wider → lower.
    sns.histplot(
        data=plot_data,
        x=col_name,
        stat='density',
        multiple='dodge',
        common_norm=False,
        kde=True,
        ax=ax,
        hue='tau_217_category',
        # palette=['#b4cde3', '#fbb4ae']  # blue for Low, red for High
        palette=['tab:blue', 'tab:red']  # blue for Low, red for High
    )
    
    
    # # Add significance marker with adjusted p-value
    # p_val = p_values_dict[feature_name]
    # if p_val is not None:
    #     # Determine significance level
    #     if p_val < 0.001:
    #         sig_marker = '***'
    #     elif p_val < 0.01:
    #         sig_marker = '**'
    #     elif p_val < 0.05:
    #         sig_marker = '*'
    #     else:
    #         sig_marker = 'ns'
        
    #     # Add text above the violins (using data coordinates)
    #     y_max = plot_data[col_name].max()
    #     y_min = plot_data[col_name].min()
    #     y_range = y_max - y_min
    #     y_position = y_max - (y_range * 0.05)  # Position 5% above max value
        
    #     # Position x at 0.5 (between Low=0 and High=1 in categorical space)
    #     ax.text(0.5, y_position, f'{sig_marker}\np = {p_val:.4f}', 
    #             ha='center', va='bottom',
    #             fontsize=11, fontweight='bold')
    
    ax.set_title(feature_name, fontsize=14, fontweight='bold')
    ax.set_xlabel(feature_name, fontsize=12)   # x-axis is the feature value
    ax.set_ylabel('Density', fontsize=12)      # y-axis is probability density
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# MOCA tests visualization

In [ ]:
df_tau['Source'].unique()

In [ ]:
df_tau = df[df['Biomarker_Type'] == 'tau_217']
df_tau_moca = df_tau[df_tau['Source'] == 'moca_tests']
df_tau_moca_all = df_tau_moca[df_tau_moca['MCI_Status'].isin(['All', 'Cognitively Healthy'])]
df_tau_moca_all.head(1)

In [ ]:
# Create a DataFrame with Features as index and adjusted p-values as values
df_plot = df_tau_moca_all.set_index('Feature')[['adjusted p-value']]
neg_log_p = df_plot['adjusted p-value'].apply(lambda x: -np.log(x))
df_plot['-log(adjusted p-value)'] = neg_log_p

# Create annotation DataFrame for stars
def star_annot(x):
    if x > -np.log(0.005):
        return '***'
    elif x > -np.log(0.01):
        return '**'
    elif x > -np.log(0.05):
        return '*'
    else:
        return ''

annot = neg_log_p.apply(star_annot).to_frame()
annot.columns = ['-log(adjusted p-value)']

plt.figure(figsize=(5, len(df_plot) * 0.5 + 1))  # dynamic height
sns.heatmap(
    data=df_plot[['-log(adjusted p-value)']],
    cmap='coolwarm',
    center=-np.log(0.1),
    annot=annot,
    vmin=0,
    vmax=-np.log(0.001),
    fmt='s',
    yticklabels=df_plot.index,
    cbar_kws={"label": "-log(Adjusted p-value)"}
)
plt.xlabel('')  # No column name on x-axis
plt.ylabel('Feature')
plt.title('MOCA Tests Adjusted p-values (tau_217)\nStars by significance (log scale) center = -log(0.1)')
plt.tight_layout()
plt.show()